[Accueil](../../index.ipynb) > [Sommaire de Terminale](../index.ipynb)

# 3.3 Base de données - Système de gestion

> **Extrait du programme**
>
> **Capacité attendue** : identifier les services rendus par un système de gestion de bases de données relationnelles : persistance des données, gestion des accès concurrents, efficacité de traitement des requêtes, sécurisation des accès.
>
> **Commentaire** : il s'agit de comprendre le rôle et les enjeux des différents services sans en détailler le fonctionnement.

---



## Objectifs

À la fin de ce chapitre, vous serez capable de :

* distinguer une **base de données** d'un **système de gestion de bases de données** (SGBD) ;
* expliquer pourquoi un simple fichier (CSV, tableur) ne suffit plus dès que les données sont nombreuses, partagées et sensibles ;
* **identifier** les quatre services rendus par un SGBD et expliquer, pour chacun, **à quoi il sert** et **quel problème il résout** :
  1. la persistance des données,
  2. la gestion des accès concurrents,
  3. l'efficacité de traitement des requêtes,
  4. la sécurisation des accès ;
* reconnaître, dans une situation donnée, le service du SGBD qui est en jeu.

> **Remarque** : on s'intéresse au **rôle** et aux **enjeux** de ces services. On ne cherche pas à savoir *comment* le SGBD les réalise en interne : c'est son travail, pas celui de l'utilisateur ni du programmeur qui l'utilise.

---



## 1. Activité d'accroche : le fichier partagé de la médiathèque

Une médiathèque gère ses prêts dans un fichier `prets.csv` stocké sur un dossier partagé du réseau. Chaque ligne contient : numéro d'adhérent, titre du livre, date de prêt, date de retour.

Les bibliothécaires ouvrent ce fichier, le modifient, puis l'enregistrent. Le site web de la médiathèque lit aussi ce fichier pour afficher la disponibilité des livres.

**Questions**

1. L'ordinateur s'éteint brutalement (coupure de courant) pendant l'enregistrement du fichier. Que peut-il se passer ?
2. Deux bibliothécaires ouvrent le fichier en même temps, chacune ajoute un prêt, puis enregistre. Quel est le risque ?
3. La médiathèque compte désormais 2 millions de lignes. Pour savoir si un livre est disponible, il faut lire le fichier en entier. Pourquoi est-ce un problème ?
4. Un stagiaire, qui n'a besoin que de consulter le catalogue, peut ouvrir le fichier et modifier ou supprimer n'importe quelle ligne. Pourquoi est-ce un problème ?

Ces quatre questions correspondent aux quatre services que rend un SGBD.

---



## 2. Rappels : base de données et SGBD

Une **base de données** est un ensemble structuré de données. Dans le modèle **relationnel**, elles sont rangées dans des **tables** reliées entre elles par des clés primaires et des clés étrangères.

Un **système de gestion de bases de données** (SGBD) est un **logiciel** chargé de gérer ces données. Les utilisateurs et les programmes ne manipulent pas directement les fichiers de données : ils envoient des **requêtes SQL** au SGBD, qui leur répond.

| SGBD | Caractéristiques |
|------|------------------|
| SQLite | Léger, sans serveur : toute la base est dans un seul fichier. Intégré au module `sqlite3` de Python. |
| PostgreSQL | Libre et très complet, utilisé dans de nombreuses organisations. |
| MariaDB / MySQL | Libres, très répandus pour les sites web. |
| Oracle, SQL Server | Propriétaires, utilisés dans de grandes entreprises. |

Dans la plupart des cas, le SGBD fonctionne comme un **serveur** auquel plusieurs **clients** se connectent.

```text
+----------+
| Client 1 |--+
+----------+  |
+----------+  |   requêtes SQL    +-----------+      +-----------------+
| Client 2 |--+------------------>|   SGBD    |<---->| fichiers de     |
+----------+  |<------------------| (serveur) |      | données (disque)|
+----------+  |     réponses      +-----------+      +-----------------+
| Client 3 |--+
+----------+
```

C'est parce que **tous les accès passent par le SGBD** que celui-ci peut rendre les services étudiés dans ce chapitre.

<div class="alert alert-info">
La base de données, ce sont les <strong>données</strong>. Le SGBD, c'est le <strong>logiciel</strong> qui les gère et les protège.
</div>

---



<img src="img/icone_persistance.svg" alt="icône persistance des données" width="50" style="float: left; margin-right: 10px;">

## 3. Service n°1 : la persistance des données

**Rôle** : conserver les données de façon durable.

**Enjeu** : ne jamais perdre ni abîmer des données validées, même en cas de panne, d'arrêt brutal ou de redémarrage de la machine.

Les variables d'un programme sont en **mémoire vive**, qui s'efface quand le programme ou la machine s'arrête. Une base de données doit au contraire survivre aux programmes qui l'utilisent : une commande passée sur un site marchand doit encore exister le lendemain, même si le serveur a redémarré entre-temps.

Le SGBD se charge de :

* enregistrer les données sur un support durable (disque, SSD...) ;
* retrouver la base dans un état **correct** après une panne : une opération terminée n'est pas perdue, une opération interrompue en cours de route n'abîme pas les données ;
* permettre de réaliser des **sauvegardes**.

Une modification devient définitive quand le programme la **valide** avec l'instruction `COMMIT`.

```python
import sqlite3

con = sqlite3.connect("mediatheque.db")   # crée le fichier s'il n'existe pas
cur = con.cursor()

cur.execute("""CREATE TABLE IF NOT EXISTS livres (
                   id INTEGER PRIMARY KEY,
                   titre TEXT,
                   auteur TEXT)""")

cur.execute("INSERT INTO livres (titre, auteur) VALUES ('Dune', 'Frank Herbert')")
con.commit()    # validation : l'insertion est désormais durable
con.close()
```

Si l'on supprime la ligne `con.commit()`, l'insertion est **perdue** à la fermeture de la connexion : elle n'a jamais été validée. Avec `commit()`, un autre programme qui ouvre `mediatheque.db` retrouve le livre : les données ont **persisté** au-delà de l'exécution du premier programme.

<div class="alert alert-info">
la persistance garantit que les données validées sont conservées durablement, même après une panne.
</div>

---



<img src="img/icone_acces_concurrents.svg" alt="icône accès concurrents" width="50" style="float: left; margin-right: 10px;">

## 4. Service n°2 : la gestion des accès concurrents

**Rôle** : permettre à plusieurs utilisateurs d'utiliser la base en même temps.

**Enjeu** : éviter que des accès simultanés rendent les données **incohérentes**.

Une base de données est rarement utilisée par une seule personne à la fois : un site marchand reçoit en permanence des demandes de nombreux clients. Que se passe-t-il quand deux clients modifient **la même donnée au même moment** ?

**Exemple** : il reste 10 exemplaires d'un article. Alice et Bob en achètent chacun un, au même instant. Chaque programme lit le stock, calcule le nouveau stock, puis l'écrit.

| Temps | Alice | Bob | Stock en base |
|-------|-------|-----|---------------|
| t1 | lit le stock : 10 | | 10 |
| t2 | | lit le stock : 10 | 10 |
| t3 | calcule 10 - 1 = 9 et écrit 9 | | 9 |
| t4 | | calcule 10 - 1 = 9 et écrit 9 | 9 |

Deux exemplaires ont été vendus, mais le stock n'a baissé que d'un seul : la base est **incohérente**.

Pour éviter cela, le SGBD gère les accès simultanés : il fait en sorte que les opérations de plusieurs utilisateurs se déroulent **sans se perturber**, quitte à faire patienter l'un d'eux.

Il s'appuie sur la notion de **transaction** : un groupe d'opérations exécuté comme un **tout indivisible**, « tout ou rien ». L'exemple classique est le virement bancaire :

```sql
BEGIN TRANSACTION;
UPDATE comptes SET solde = solde - 100 WHERE id = 1;   -- débit
UPDATE comptes SET solde = solde + 100 WHERE id = 2;   -- crédit
COMMIT;
```

* `COMMIT` **valide** la transaction : les deux opérations sont appliquées.
* `ROLLBACK` **annule** la transaction : on revient à l'état précédent.

Si un problème survient entre le débit et le crédit, la transaction est annulée : on n'a jamais un compte débité sans que l'autre soit crédité.

<div class="alert alert-info">
le SGBD permet à de nombreux utilisateurs de travailler simultanément sur les mêmes données <strong>sans produire d'incohérences</strong>. Les transactions garantissent qu'un ensemble d'opérations est exécuté entièrement ou pas du tout.
</div>

---



<img src="img/icone_efficacite.svg" alt="icône efficacité" width="50" style="float: left; margin-right: 10px;">

## 5. Service n°3 : l'efficacité de traitement des requêtes

**Rôle** : répondre rapidement aux requêtes, même avec de très gros volumes de données et de nombreux utilisateurs.

**Enjeu** : conserver un temps de réponse acceptable. Un site dont les pages mettent 30 secondes à s'afficher est inutilisable.

Pour retrouver un élève par son nom dans une table de 500 000 lignes, la méthode la plus simple est de parcourir **toutes** les lignes. Plus la table est grosse, plus c'est long (question 3 de l'activité).

Le SGBD dispose pour cela de deux atouts :

* les **index** : un index est comparable à l'index à la fin d'un livre. Pour trouver une notion, on ne lit pas tout le livre : on consulte l'index, qui indique directement les pages. On demande au SGBD d'en créer un sur une colonne avec `CREATE INDEX` ;
* l'**optimisation des requêtes** : une requête SQL décrit **ce que l'on veut**, pas **comment l'obtenir**. C'est le SGBD qui choisit lui-même la façon la plus efficace de la traiter (par exemple en utilisant un index quand il en existe un).

```sql
CREATE INDEX idx_nom ON eleves (nom);
```

> **Remarque** : un index n'est pas gratuit : il occupe de la place et rend les écritures un peu plus lentes. On l'ajoute sur les colonnes souvent utilisées pour chercher, pas sur toutes.

**Observer l'effet d'un index**

```python
import sqlite3, random, string, time

con = sqlite3.connect(":memory:")
cur = con.cursor()
cur.execute("CREATE TABLE eleves (id INTEGER PRIMARY KEY, nom TEXT, classe TEXT)")

def nom_aleatoire():
    return "".join(random.choice(string.ascii_lowercase) for _ in range(8))

cur.executemany("INSERT INTO eleves (nom, classe) VALUES (?, ?)",
                [(nom_aleatoire(), "TNSI") for _ in range(200_000)])
con.commit()

def duree(requete):
    debut = time.perf_counter()
    cur.execute(requete).fetchall()
    return time.perf_counter() - debut

requete = "SELECT * FROM eleves WHERE nom = 'abcdefgh'"
print("Sans index :", duree(requete))

cur.execute("CREATE INDEX idx_nom ON eleves (nom)")
print("Avec index :", duree(requete))
```

La requête est la même dans les deux cas : seul le travail du SGBD change, et la réponse arrive beaucoup plus vite avec l'index.

<div class="alert alert-info">
le SGBD est conçu pour traiter efficacement les requêtes, en particulier grâce aux index et à l'optimisation qu'il réalise lui-même.
</div>

---



<img src="img/icone_securite.svg" alt="icône sécurité" width="50" style="float: left; margin-right: 10px;">

## 6. Service n°4 : la sécurisation des accès

**Rôle** : contrôler **qui** accède aux données et **ce qu'il a le droit d'en faire**.

**Enjeu** : protéger des données souvent sensibles (notes, informations personnelles, données bancaires) contre les accès non autorisés, les modifications abusives et les erreurs (question 4 de l'activité).

Le SGBD propose notamment :

* l'**authentification** : l'utilisateur doit prouver son identité (nom d'utilisateur et mot de passe, par exemple) avant de pouvoir se connecter ;
* la **gestion des droits** : chaque utilisateur ou groupe d'utilisateurs reçoit uniquement les droits dont il a besoin (lire, ajouter, modifier, supprimer) sur chaque table. Par exemple, une secrétaire peut consulter les élèves, un professeur peut aussi saisir des notes, un stagiaire ne voit qu'une partie des informations ;
* la possibilité de **restreindre** ce qui est visible, en ne montrant à certains utilisateurs qu'une partie des données.

```sql
GRANT SELECT ON eleves TO secretariat;          -- lecture seule
GRANT SELECT, INSERT, UPDATE ON notes TO prof;  -- lecture et écriture
```

On applique le **principe du moindre privilège** : chacun ne reçoit que les droits strictement nécessaires à sa mission.

> **Remarque** : SQLite n'a pas de gestion d'utilisateurs : la protection repose alors sur les droits d'accès au fichier par le système d'exploitation. Les droits détaillés se rencontrent dans les SGBD de type serveur (PostgreSQL, MariaDB...).

> **Attention** : le SGBD ne peut pas tout : une **application mal écrite** peut ouvrir une faille. Dans une **injection SQL**, un utilisateur malveillant saisit du code SQL dans un formulaire pour détourner la requête prévue. On s'en protège en ne construisant jamais une requête par concaténation de chaînes saisies, mais avec des **requêtes paramétrées** :
>
> ```python
> cur.execute("SELECT * FROM utilisateurs WHERE login = ? AND mdp = ?",
>             (login, mdp))
> ```

<div class="alert alert-info">
La sécurisation des accès permet de réserver les données aux personnes autorisées et de limiter ce que chacun peut faire.
</div>

---



## 7. Synthèse

| Service | Enjeu | Rôle du SGBD | Exemple |
|---------|-------|--------------|---------|
| <img src="img/icone_persistance.svg" alt="icône persistance des données" width="30" style="float: left; margin-right: 10px;"> **Persistance** | Ne pas perdre ni abîmer les données validées | Conserve les données durablement, même après une panne | Commandes retrouvées après un redémarrage du serveur |
| <img src="img/icone_acces_concurrents.svg" alt="icône accès concurrents" width="30" style="float: left; margin-right: 10px;"> **Accès concurrents** | Éviter les incohérences quand plusieurs utilisateurs agissent en même temps | Gère les accès simultanés, exécute les transactions « tout ou rien » | Dernière place d'un concert vendue une seule fois |
| <img src="img/icone_efficacite.svg" alt="icône efficacité" width="30" style="float: left; margin-right: 10px;"> **Efficacité des requêtes** | Garder des temps de réponse acceptables | Retrouve vite les données (index) et choisit lui-même comment traiter la requête | Recherche d'un client parmi des millions |
| <img src="img/icone_securite.svg" alt="icône sécurité" width="30" style="float: left; margin-right: 10px;"> **Sécurisation des accès** | Protéger les données sensibles | Authentifie les utilisateurs et contrôle leurs droits | Un élève consulte ses notes mais ne peut pas les modifier |


<div class="alert alert-info">Un SGBD veille aussi à l'<strong>intégrité</strong> des données grâce aux contraintes qu'on lui déclare (clé primaire, clé étrangère, `NOT NULL`...) : il refuse une insertion qui rendrait la base incohérente.
</div>

---



## 8. Vocabulaire

* **Base de données** : ensemble structuré de données.
* **SGBD** : logiciel qui gère une base de données et en contrôle les accès.
* **Persistance** : conservation durable des données validées, même après une panne.
* **Accès concurrents** : accès simultanés de plusieurs utilisateurs aux mêmes données.
* **Transaction** : groupe d'opérations exécuté comme un tout indivisible : tout est appliqué (`COMMIT`) ou rien (`ROLLBACK`).
* **Index** : structure qui permet de retrouver rapidement des données dans une table.
* **Droit (ou privilège)** : autorisation accordée à un utilisateur sur une table ou une partie de la base.
* **Injection SQL** : attaque consistant à saisir du code SQL pour détourner une requête.

---


[Accueil](../../index.ipynb) > [Sommaire de Terminale](../index.ipynb)